In [9]:
from pathlib import Path
import optuna
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import  StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    classification_report,
)

current_path = Path.cwd()

project_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").exists()
)

data_path = (
    project_root
    / "data"
    / "preprocessing"
    / "train_processed.csv"
)

dataset = pd.read_csv(data_path, index_col=0)

X = dataset.drop(columns=["Attrition"])
y = dataset["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

def objective(trial):
    params = {
            "n_estimators": trial.suggest_int(
                "n_estimators", 100, 1000, step=50
            ),
            "max_depth": trial.suggest_int(
                "max_depth", 2, 5
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.2, log=True
            ),
            "subsample": trial.suggest_float(
                "subsample", 0.8, 1.0
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.7, 0.9
            ),
            "min_child_weight": trial.suggest_int(
                "min_child_weight", 1, 10
            ),
            "gamma": trial.suggest_float(
                "gamma", 0.0, 5.0
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha", 1e-8, 10.0, log=True
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 1e-8, 10.0, log=True
            ),

            # 고정 설정
           "max_leaves": 0,
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "tree_method": "hist",
            "device": "cuda",
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": 0,
        }

    model = XGBClassifier(**params)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,  # GPU 한 장에 여러 모델을 동시에 올리지 않는다.
    )

    return scores.mean()

study = optuna.create_study(
    direction="maximize", 
)

study.optimize(
    objective,
    n_trials=50,
    timeout=600
)

[I 2026-08-25 17:42:43,645] A new study created in memory with name: no-name-862da159-32ed-47ed-8ef0-f6984ee8beea
[I 2026-08-25 17:42:48,095] Trial 0 finished with value: 0.8483758962198982 and parameters: {'n_estimators': 250, 'max_depth': 3, 'learning_rate': 0.0458952988854849, 'subsample': 0.9496198020908537, 'colsample_bytree': 0.8985504669900463, 'min_child_weight': 7, 'gamma': 3.688005526324773, 'reg_alpha': 9.834456416732134e-08, 'reg_lambda': 0.0015242626017324685}. Best is trial 0 with value: 0.8483758962198982.
[I 2026-08-25 17:42:56,344] Trial 1 finished with value: 0.8506121489854127 and parameters: {'n_estimators': 600, 'max_depth': 2, 'learning_rate': 0.07596530523294677, 'subsample': 0.8619664952279417, 'colsample_bytree': 0.8627000289678488, 'min_child_weight': 2, 'gamma': 2.718421086614138, 'reg_alpha': 0.0013857198435202154, 'reg_lambda': 6.425594054921586e-07}. Best is trial 1 with value: 0.8506121489854127.
[I 2026-08-25 17:43:03,600] Trial 2 finished with value: 0.

In [10]:
trial = study.best_trial

print("Value:", trial.value)
print("Params:", trial.params)

Value: 0.8508399017209328
Params: {'n_estimators': 250, 'max_depth': 2, 'learning_rate': 0.13498818933282644, 'subsample': 0.800257477114954, 'colsample_bytree': 0.7971425692129515, 'min_child_weight': 8, 'gamma': 3.9998551427843294, 'reg_alpha': 0.0006661254916300219, 'reg_lambda': 2.035969159490572e-08}
